# PM-RL1 离线账本复算
仅从最终私人输出读取机械字段，不打开原始 mixed/test 容器，不加载模型。报告口径是候选位置/已测对照，不是质量金标。应从仓库或其子目录运行。

In [1]:
from pathlib import Path
import json, hashlib
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "project/src/metacom_pm/rl1").is_dir())
run = root / "project/outputs/pm_rl1/reorganization_20261029_v4"
def read(name):
    return json.loads((run / name).read_text())
manifest = read("OUTPUT_MANIFEST.json")
assert all(hashlib.sha256((run / p).read_bytes()).hexdigest() == h for p, h in manifest.items())
rows = read("CANDIDATE_LEDGER_PRIVATE.json")
capacity = read("CAPACITY_PRIVATE.json")
print({"manifest_files_verified": len(manifest), "prefixes": len(capacity), "candidate_occurrences": len(rows)})


{'manifest_files_verified': 245, 'prefixes': 114, 'candidate_occurrences': 1704}


In [2]:
from collections import Counter
for head in ("RS", "MP", "MS", "ME"):
    selected = [r for r in rows if r["head"] == head]
    print(head, {"candidates": len(selected), "reachable": sum(r["reachable_in_any_plan"] for r in selected),
                 "source_checks": dict(Counter(r["source_check"] for r in selected))})
case = next(r for r in rows if r["index"] == 0 and r["head"] == "MS" and r["rank"] == 3)
print("MS example", {k: case[k] for k in ("index", "rank", "hypothetical_isolated_tokens", "prefix_resource_tokens", "reachable_in_any_plan")})
assert len(read("DOMINATED_PRIVATE.json")) == 52
assert len(read("RESPONSE_LEDGER_PRIVATE.json")) == 586
assert len(read("EXECUTOR_PAIRS_PRIVATE.json")) == 53
print({"same_content_higher_cost": 52, "matched_add_one_edges": len(read("MATCHED_CONTRASTS_PRIVATE.json"))})


RS {'candidates': 456, 'reachable': 456, 'source_checks': {'canonical_alias_text_and_all_train_ids_verified': 456}}
MP {'candidates': 374, 'reachable': 374, 'source_checks': {'exact_spans_roles_and_offsets_verified': 320, 'unverified': 54}}
MS {'candidates': 425, 'reachable': 246, 'source_checks': {'complete_raw_session_roles_and_text_verified': 354, 'unverified': 71}}
ME {'candidates': 449, 'reachable': 449, 'source_checks': {'exact_spans_roles_and_offsets_verified': 380, 'unverified': 69}}
MS example {'index': 0, 'rank': 3, 'hypothetical_isolated_tokens': 951, 'prefix_resource_tokens': 2152, 'reachable_in_any_plan': False}
{'same_content_higher_cost': 52, 'matched_add_one_edges': 593}


In [3]:
branching_prefixes = 0
states_with_continuation = 0
for prefix in capacity:
    reachable = {tuple(r["counts"]) for r in prefix["plans"] if r["reachable"]}
    branch_counts = []
    for counts in reachable:
        if not 0 < sum(counts) < 4:
            continue
        alternatives = sum(tuple(n + (i == j) for i, n in enumerate(counts)) in reachable for j in range(4))
        if alternatives:
            branch_counts.append(alternatives)
    states_with_continuation += len(branch_counts)
    branching_prefixes += any(n >= 2 for n in branch_counts)
print({"prefixes_with_multiple_GET_choices_after_GET": branching_prefixes,
       "legal_post_GET_states_with_STOP_and_GET": states_with_continuation,
       "content_conditional_quality_gain": "not_established"})
assert branching_prefixes == 114 and states_with_continuation == 3566


{'prefixes_with_multiple_GET_choices_after_GET': 114, 'legal_post_GET_states_with_STOP_and_GET': 3566, 'content_conditional_quality_gain': 'not_established'}


这些统计证明合法动作/来源/成本结构，不证明资源有用、G正确采纳或J合格。已知人工和误用案例见私人case ledger，未用机械匹配补标。交互命令曾输出mixed原文容器的暴露事件另见EXPOSURE_INCIDENT.json；本notebook不读取它。